# Zero Mouse Clicks — train the floor-plan reader on a free GPU (Kaggle / Colab)

Trains the small U-Net that reads marketing floor plans **using synthetic plans only** (`zmc/synth.py`), so the weights carry no third-party dataset license. Exports ONNX models that run on CPU or in the browser (onnxruntime-web).

**How to run**
- **Kaggle**: *Settings → Accelerator → GPU T4 x2*, *Internet on*, then *Save Version → Save & Run All (Commit)*.
- **Colab**: *Runtime → Change runtime type → T4 GPU*, then *Run all*.
- Download `zmc_unet_fp32.onnx` / `zmc_unet_fp16.onnx` from the output panel and copy the fp32 file to `zmc/models/unet_v4.onnx` to try it with the pipeline.

**Recipe of the published `unet_v4` weights**: 512 px crops, 20 epochs × 600 steps, batch 20 on 2× T4, fp32, about 325 k parameters, 8 classes:
`0 background · 1 space · 2 wall · 3 door · 4 window · 5 counter/closet/bar · 6 fixed fixture · 7 loose furniture`.

Notes from the runs: mixed precision fails under XLA with bilinear upsampling, so training is fp32. The bottleneck is the CPU generating plans, not the GPU: pre-generating a dataset is the easiest speed-up. The code comments in `zmc/` are in Spanish (the project started in Colombia).

In [ ]:
!pip -q install tf2onnx onnx onnxconverter-common onnxruntime opencv-python-headless
!git clone -q https://github.com/zeromouseclicks/zero-mouse-clicks.git
import sys, os, time, numpy as np, tensorflow as tf
sys.path.insert(0, 'zero-mouse-clicks')
from zmc import synth
print(tf.__version__, tf.config.list_physical_devices('GPU'))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 6, figsize=(18, 6))
for i in range(6):
    im, m, st = synth.render(512)
    ax[0, i].imshow(im[..., ::-1]); ax[0, i].set_title(st); ax[1, i].imshow(synth.colorize(m)[..., ::-1])
    ax[0, i].axis('off'); ax[1, i].axis('off')
plt.show()

## Configuration

In [ ]:
SIZE = 512                     # training crop
EPOCHS, STEPS_PER_EPOCH = 20, 600
FILTERS = (24, 48, 96, 160, 256)
NC = 8
CLASS_NAMES = ['background', 'space', 'wall', 'door', 'window', 'counter/closet', 'fixture', 'furniture']
gpus = tf.config.list_physical_devices('GPU')
strategy = tf.distribute.MirroredStrategy() if len(gpus) > 1 else tf.distribute.get_strategy()
BATCH = 10 * max(1, strategy.num_replicas_in_sync)   # 20 on 2x T4
print('replicas', strategy.num_replicas_in_sync, 'batch', BATCH)

## Synthetic data on the fly (several CPU processes)

In [ ]:
import multiprocessing as mp
def _worker(q, seed):
    import sys, numpy as np
    sys.path.insert(0, 'zero-mouse-clicks')
    from zmc import synth
    synth.RNG = np.random.default_rng(seed)
    while True:
        im, m, _ = synth.render(SIZE); q.put((im, m))
Q = mp.Queue(maxsize=256)
procs = [mp.Process(target=_worker, args=(Q, s), daemon=True) for s in range(max(1, os.cpu_count() - 1))]
for p in procs: p.start()
def gen():
    while True:
        yield Q.get()
def aug(x, y):
    flip = tf.random.uniform(()) < 0.5
    x = tf.cond(flip, lambda: tf.image.flip_left_right(x), lambda: x)
    y = tf.cond(flip, lambda: tf.image.flip_left_right(y[..., None])[..., 0], lambda: y)
    flip = tf.random.uniform(()) < 0.5
    x = tf.cond(flip, lambda: tf.image.flip_up_down(x), lambda: x)
    y = tf.cond(flip, lambda: tf.image.flip_up_down(y[..., None])[..., 0], lambda: y)
    return x, y
ds = (tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((SIZE, SIZE, 3), tf.uint8), tf.TensorSpec((SIZE, SIZE), tf.uint8)))
      .map(aug, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE))
Xv, Yv = zip(*[synth.render(SIZE)[:2] for _ in range(200)]); Xv, Yv = np.stack(Xv), np.stack(Yv)
freq = np.bincount(Yv.ravel(), minlength=NC) / Yv.size
cw = np.clip((1 / (freq + 1e-6)) ** 0.5, 1, None); cw = cw / cw.min(); print('class weights', cw.round(2))

## Model: U-Net with separable convolutions

In [ ]:
from tensorflow.keras import layers as L
def block(x, f):
    for _ in range(2):
        x = L.SeparableConv2D(f, 3, padding='same', use_bias=False)(x)
        x = L.BatchNormalization()(x); x = L.ReLU()(x)
    return x
def unet(fs=FILTERS):
    inp = L.Input((None, None, 3))
    x = L.Rescaling(1 / 127.5, offset=-1)(inp)
    x = L.Conv2D(fs[0], 3, padding='same')(x); skips = []
    for f in fs[:-1]:
        x = block(x, f); skips.append(x); x = L.MaxPool2D()(x)
    x = block(x, fs[-1])
    for f, s in zip(reversed(fs[:-1]), reversed(skips)):
        x = L.UpSampling2D(interpolation='bilinear')(x); x = L.Concatenate()([x, s]); x = block(x, f)
    out = L.Conv2D(NC, 1, dtype='float32')(x)
    return tf.keras.Model(inp, out)
with strategy.scope():
    model = unet(); print(model.count_params(), 'parameters')
    cw_t = tf.constant(cw, tf.float32)
    def loss(y, logits):
        y = tf.cast(y, tf.int32)
        ce = tf.keras.losses.sparse_categorical_crossentropy(y, logits, from_logits=True)
        return tf.reduce_mean(ce * tf.gather(cw_t, y))
    steps = EPOCHS * STEPS_PER_EPOCH
    lr = tf.keras.optimizers.schedules.CosineDecay(0.0, steps, warmup_target=3e-3, warmup_steps=500)
    model.compile(tf.keras.optimizers.AdamW(lr, weight_decay=1e-4), loss=loss)

## Train

In [ ]:
ck = tf.keras.callbacks.ModelCheckpoint('zmc_unet.keras')
hist = model.fit(ds, epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH, validation_data=(Xv, Yv), callbacks=[ck], verbose=2)

## Synthetic evaluation (IoU per class)

In [ ]:
pred = np.argmax(model.predict(Xv, batch_size=8, verbose=0), -1)
for c, n in enumerate(CLASS_NAMES):
    i = np.logical_and(pred == c, Yv == c).sum(); u = np.logical_or(pred == c, Yv == c).sum()
    print(f'IoU {n:15s} {i / max(u, 1):.3f}')

## Export to ONNX (fp32 and fp16), fixed input 1×768×768×3
The pipeline pads the image with white up to that size.

In [ ]:
import tf2onnx, onnx
from onnxconverter_common import float16
m32 = unet(); m32.set_weights(model.get_weights())
spec = (tf.TensorSpec((1, 768, 768, 3), tf.float32, name='image'),)
tf2onnx.convert.from_keras(m32, input_signature=spec, opset=17, output_path='zmc_unet_fp32.onnx')
onnx.save(float16.convert_float_to_float16(onnx.load('zmc_unet_fp32.onnx'), keep_io_types=True), 'zmc_unet_fp16.onnx')
for f in ['zmc_unet_fp32.onnx', 'zmc_unet_fp16.onnx']:
    print(f, round(os.path.getsize(f) / 1e6, 2), 'MB')

In [ ]:
import onnxruntime as ort
s = ort.InferenceSession('zmc_unet_fp16.onnx')
x = np.full((1, 768, 768, 3), 255, np.float32); x[0, :SIZE, :SIZE] = Xv[0]
p = np.argmax(s.run(None, {s.get_inputs()[0].name: x})[0][0], -1)[:SIZE, :SIZE]
print('pixel accuracy vs. label:', (p == Yv[0]).mean())
for p_ in procs: p_.terminate()